# Chương 2 – Quickstart LoRA 5 bước (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter02/chapter02_colab.ipynb)

1. **Listing 2.1** – gọi managed API (Claude) — phía "buy" của quyết định buy-vs-build.
2. **Quickstart §2.8** – fine-tune LoRA Qwen3-4B trên 40 ví dụ IT-support (~10 phút trên L4).
3. **Preview adapter chương 5** – so sánh base vs adapter đã train sẵn trên Hugging Face.

**GPU bắt buộc cho bước train:** L4 hoặc A100 (T4 không hỗ trợ bf16).

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

## 1. Listing 2.1 – Gọi managed API (Claude)

Tương đương `python -m chapter02.managed_api_call`, viết lại để đọc đúng khối text trong response (model mới có thể trả về khối thinking trước khối text).

In [ ]:
import anthropic

client = anthropic.Anthropic()
message = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1000,
    messages=[{"role": "user",
               "content": "Explain the difference between a stock and a bond in two sentences."}],
)
print("".join(b.text for b in message.content if b.type == "text"))

## 2. Dữ liệu IT-support

Dữ liệu đã được commit sẵn trong `data/it_support` và `data/it_support_fmt`, nên **không cần** build lại. Ô dưới chỉ kiểm tra file. (Muốn tự build lại từ Stack Exchange: `python scripts/build_it_support_dataset.py` — mất khá lâu.)

In [ ]:
run("ls -la data/it_support data/it_support_fmt && wc -l data/it_support_fmt/*.jsonl data/it_support/*.jsonl")

## 3. Chạy quickstart LoRA 5 bước

Chuẩn bị dữ liệu → load model + LoRA → train 20 bước → so sánh output → lưu adapter vào `chapter02/runs/ch2_quickstart/`.

In [ ]:
run("python -m chapter02.quickstart")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter02/runs/ch2_quickstart/manifest.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000] if p.exists() else f"Chưa có {p}")

## 4. (Tuỳ chọn) Xem trước adapter của chương 5

Script tự tìm adapter: local `chapter05/runs/it_lora/` → Hugging Face Hub.

In [ ]:
run("python -m chapter02.run_chapter5_adapter")

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter02/runs"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)